# Tutorial 04 - Exercise 03 (Part 2): Hospital Risk Scoring & Pivot Analytics

Computes total costs, patient age segments, composite clinical risk scores,
pivot aggregations, and regional exports. It continues notebook 01 (indexing
and slicing) on the same 8-patient cohort and completes the remaining tasks
of the Exercise 03 handout (Tasks 3, 4, 5, 9 and 10).

---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/06_Hospital_Department_Analytics/` project folder (anchored on `data/hospital_original.csv`; exports also go to the project's `data/` folder) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "06_Hospital_Department_Analytics"
ANCHOR_REL = "data/hospital_original.csv"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Total Cost & Age Groups


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(PROJ_DIR / "data" / "hospital_original.csv")

df["TotalCost"] = df["DaysAdmitted"] * df["DailyCost"]

def categorize_age(age):
    if age < 30: return "Young"
    elif 30 <= age <= 50: return "Middle"
    else: return "Senior"

df["AgeGroup"] = df["Age"].apply(categorize_age)
df = df.rename(columns={"DailyCost": "DailyFee"})
display(df.head())


### 2. Clinical Risk Score Computation

$$\text{RiskScore} = (\text{DaysAdmitted} \times 0.4) + (\text{Age} \times 0.3) + (100 - \text{Satisfaction} \times 10) \times 0.3$$


In [ ]:
df["RiskScore"] = (df["DaysAdmitted"] * 0.4) + (df["Age"] * 0.3) + (100 - df["Satisfaction"] * 10) * 0.3

def categorize_risk(score):
    if score > 60: return "High"
    elif 40 <= score <= 60: return "Medium"
    else: return "Low"

df["RiskCategory"] = df["RiskScore"].apply(categorize_risk)
df = df.sort_values(by="RiskScore", ascending=False)
display(df[["City", "Age", "DaysAdmitted", "Satisfaction", "RiskScore", "RiskCategory"]])


### 3. City Summary Aggregations


In [ ]:
city_summary = df.groupby("City").agg(
    AvgTotalCost=("TotalCost", "mean"),
    AvgSatisfaction=("Satisfaction", "mean"),
    ReadmissionRate=("Readmitted", "mean"),
).reset_index()

display(city_summary)

highest_risk = df.groupby("City")["RiskScore"].mean().idxmax()
print(f"\nHighest Clinical Risk City: {highest_risk}")


### 4. Data Mapping & Pivot Tables


In [ ]:
df["RegionCode"] = df["City"].map({"Alger": "DZ-16", "Annaba": "DZ-23", "Oran": "DZ-31"})
df["ReadmissionStatus"] = df["Readmitted"].map({0: "No", 1: "Yes"})

pivot = df.pivot_table(index="City", columns="Department", values=["TotalCost", "Satisfaction"], aggfunc="mean")
display(pivot)

# Export processed tables (into the project data folder)
df.to_csv(PROJ_DIR / "data" / "hospital_risk.csv", index=False)
city_summary.to_csv(PROJ_DIR / "data" / "city_summary.csv", index=False)
print("Processed tables exported to CSV.")


### 5. Task Map (Exercise 03, Tasks 1-11)

| Task | What the handout asks | Where it is done |
| :--- | :--- | :--- |
| 1 | Total cost of each stay (`TotalCost`) | Section 1 |
| 2 | Age groups (`AgeGroup`) | Section 1 |
| 3 | Bonus of 10%: add, rename, then drop | Section 6 |
| 4 | `HighCost` and `SatisfactionLevel` flags | Section 7 |
| 5 | `CostRank` | Section 8 |
| 6 | Clinical risk score and category | Section 2 |
| 7 | Highest-risk city | Section 3 |
| 8 | Rename `DailyCost` to `DailyFee` | Section 1 |
| 9 | Impute a missing satisfaction before the level mapping | Section 9 |
| 10 | `pivot_table` versus `groupby` | Section 10 |
| 11 | Region codes, readmission status, pivot table, CSV exports | Section 4 |

The handout lists its instructions without a task number next to every one of them. The numbering
above follows their order and is an inference, recorded in repository issue #14; the content of
each task is the handout's.

### 6. Task 3: Bonus Column (Add, Rename, Drop)

The bonus is 10% of the total cost. It is added under its working name, renamed, then removed, so
the exported tables keep exactly the columns of the previous sections. A copy of the frame is used
to leave the working frame untouched.

In [ ]:
df_bonus = df.copy()
df_bonus["Bonus"] = df_bonus["TotalCost"] * 0.10
df_bonus = df_bonus.rename(columns={"Bonus": "LoyaltyBonus"})
display(df_bonus[["City", "TotalCost", "LoyaltyBonus"]].head(3))

df_bonus = df_bonus.drop(columns=["LoyaltyBonus"])
print("LoyaltyBonus still present after the drop:", "LoyaltyBonus" in df_bonus.columns)
print("working frame unchanged:", "Bonus" in df.columns)

### 7. Task 4: `HighCost` and `SatisfactionLevel` (Handout Thresholds)

> **Handout thresholds, kept as written.** The handout flags a stay as high cost when
> `TotalCost > 60,000`, and rates the satisfaction `Good` when `Satisfaction >= 8`, `Average`
> from 5 up to 8, and `Poor` below 5. The cohort does not live on those scales: `TotalCost`
> runs from 420 to 2610 and `Satisfaction` from 3.5 to 4.7. The instructor's logic is applied
> unchanged first - and puts every stay on the same side of both thresholds. The same logic is
> then rerun with thresholds read from the data, clearly labelled as a sensitivity check.

In [ ]:
def satisfaction_level_handout(score):
    if score >= 8:
        return "Good"
    elif score >= 5:
        return "Average"
    return "Poor"


df["HighCost"] = df["TotalCost"].apply(lambda cost: "Yes" if cost > 60_000 else "No")
df["SatisfactionLevel"] = df["Satisfaction"].apply(satisfaction_level_handout)

print("TotalCost range   :", int(df["TotalCost"].min()), "to", int(df["TotalCost"].max()))
print("Satisfaction range:", df["Satisfaction"].min(), "to", df["Satisfaction"].max())
print("HighCost          :", df["HighCost"].value_counts().to_dict())
print("SatisfactionLevel :", df["SatisfactionLevel"].value_counts().to_dict())
display(df[["City", "TotalCost", "HighCost", "Satisfaction", "SatisfactionLevel"]])

**Sensitivity check (data-scaled thresholds - not the handout's).** The upper quartile of
`TotalCost` is 1755.0; 2 stays lie above it. Cutting the satisfaction
into three equal-frequency groups gives {'Poor': 3, 'Average': 2, 'Good': 3}.
Both columns become informative, and neither is the handout's definition: they exist to show that
the classification depends entirely on where the thresholds sit.

In [ ]:
high_cost_threshold = df["TotalCost"].quantile(0.75)
df["HighCost_DataScaled"] = np.where(df["TotalCost"] > high_cost_threshold, "Yes", "No")
df["SatisfactionBand_DataScaled"] = pd.qcut(df["Satisfaction"], 3, labels=["Poor", "Average", "Good"])

print(f"data-scaled high-cost threshold (75th percentile): {high_cost_threshold:.1f}")
print("HighCost_DataScaled        :", df["HighCost_DataScaled"].value_counts().to_dict())
print("SatisfactionBand_DataScaled:", df["SatisfactionBand_DataScaled"].value_counts().sort_index().to_dict())

### 8. Task 5: `CostRank`

The handout names only the column. Two granularities are shown: each stay ranked among the eight
stays, and each city ranked by its average total cost (Annaba 1680, Oran 1393, Alger 987).

In [ ]:
df["CostRank"] = df["TotalCost"].rank(ascending=False, method="min").astype(int)
display(df[["City", "Department", "TotalCost", "CostRank"]])

city_costs = df.groupby("City")["TotalCost"].mean().sort_values(ascending=False)
city_cost_rank = city_costs.rank(ascending=False, method="min").astype(int).rename("CostRank")
display(city_cost_rank.to_frame())
print("costliest city by average:", city_costs.idxmax())

### 9. Task 9: Missing Satisfaction - Impute Before Mapping

A missing satisfaction is dangerous precisely because nothing fails: `NaN >= 8` and `NaN >= 5`
are both `False`, so the mapping silently files the stay under `Poor`. The order matters - impute
first, then map. One satisfaction is blanked in a copy to demonstrate both orders.

In [ ]:
df_missing = df.copy()
target = df_missing.index[2]
df_missing.loc[target, "Satisfaction"] = np.nan

print("missing before imputation:", int(df_missing["Satisfaction"].isna().sum()))
print("level assigned WITHOUT imputing:", df_missing["Satisfaction"].apply(satisfaction_level_handout).loc[target])

median_satisfaction = df_missing["Satisfaction"].median()
df_missing["Satisfaction"] = df_missing["Satisfaction"].fillna(median_satisfaction)
print(f"median used for imputation: {median_satisfaction}")
print("missing after imputation :", int(df_missing["Satisfaction"].isna().sum()))
print("level assigned WITH imputing :", df_missing["Satisfaction"].apply(satisfaction_level_handout).loc[target])

### 10. Task 10: `pivot_table` versus `groupby`

Both apply split-apply-combine; they differ in the shape of the result and in the shortcuts they offer.

| Criterion | `pivot_table` | `groupby` |
| :--- | :--- | :--- |
| Result | A wide table: one row per `index` value, one column per `columns` value and statistic | Long groups, reshaped by a later `.unstack()` |
| Several statistics | `aggfunc=["mean", "sum"]` in one call | `.agg(["mean", "sum"])` |
| Missing combinations | A `NaN` cell keeps the grid complete | The combination is simply absent |
| Grand total | `margins=True` | `.agg` on the whole frame, separately |
| Default statistic | `mean` | none until an aggregation is chosen |

Use `pivot_table` when a rectangular report is the goal (a spreadsheet-like summary, complete
grid). Use `groupby` for sequential analysis - filter, transform, several different reductions -
because the grouped object stays available. On a single statistic the two return the same numbers:

In [ ]:
by_pivot = df.pivot_table(index="City", values="TotalCost", aggfunc="mean")
by_group = df.groupby("City")["TotalCost"].mean().to_frame()

display(by_pivot.round(2))
print("same index:", by_pivot.index.equals(by_group.index))
print("same values:", bool(np.allclose(by_pivot["TotalCost"], by_group["TotalCost"])))
print("missing-combination behaviour: pivot_table ->", df.pivot_table(index="City", columns="Department", values="TotalCost").isna().sum().sum(),
      "NaN cell(s); groupby size ->", int(df.groupby(["City", "Department"]).size().size), "combination(s)")